# Assignment 2: Advanced Pandas and Data Wrangling - Solution

**Name:** Hamzat Tiamiyu Ustaz

---
## Table of Contents
1. Importing Libraries
2. Loading the Datasets
3. Exploring the Data
4. Cleaning the Transactions Dataset
5. Cleaning the Prices Dataset
6. Merging the Datasets
7. Handling Missing Values in Merged Data
8. Reshaping the Dataset
9. Final Data Validation
10. Exporting the Cleaned Dataset

---
## 1. Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Libraries imported successfully!
Pandas version: 3.0.2
NumPy version: 2.4.4


---
## 2. Loading the Datasets

In [2]:
# Load the transaction data
# Note: Update the path if your CSV files are in a different location
transactions = pd.read_csv('nigerian_market_transactions_1000.csv')

# Load the weekly product prices data
weekly_prices = pd.read_csv('nigerian_weekly_product_prices_1000.csv')

print("=== TRANSACTIONS DATASET ===")
print(f"Shape: {transactions.shape}")
print(f"Columns: {list(transactions.columns)}\n")

print("=== WEEKLY PRICES DATASET ===")
print(f"Shape: {weekly_prices.shape}")
print(f"Columns: {list(weekly_prices.columns)}")

=== TRANSACTIONS DATASET ===
Shape: (1000, 10)
Columns: ['transaction_id', 'transaction_date', 'market', 'state', 'product', 'category', 'quantity', 'unit_price', 'total_amount', 'vendor']

=== WEEKLY PRICES DATASET ===
Shape: (1000, 5)
Columns: ['product', 'category', 'market', 'week_start_date', 'avg_price']


---
## 3. Exploring the Data

In [3]:
# Explore Transactions Dataset
print("=== TRANSACTIONS DATASET OVERVIEW ===\n")

print("First 5 rows:")
print(transactions.head())
print("\nData Types:")
print(transactions.dtypes)
print("\nMissing Values:")
print(transactions.isnull().sum())
print("\nDuplicated Rows:", transactions.duplicated().sum())

=== TRANSACTIONS DATASET OVERVIEW ===

First 5 rows:
  transaction_id transaction_date   market  state      product category  \
0        TXN0001       2024/01/24  Balogun  Lagos    Rice 50kg   Grains   
1        TXN0002       2024/01/17   Bodija    Oyo    Rice 50kg   Grains   
2        TXN0003              NaN   Bodija    Oyo  Beans brown   Grains   
3        TXN0004       2024-01-25   Bodija    Oyo  Garri white   Grains   
4        TXN0005       2024/01/02  Balogun  Lagos  Garri white   Grains   

   quantity  unit_price  total_amount       vendor  
0       5.0     78006.0      390030.0  Alhaji Musa  
1       1.0     76767.0       76767.0   Mama Ngozi  
2      10.0      7127.0       71270.0   Mama Ngozi  
3       5.0      2673.0       13365.0       Mr Ade  
4       5.0      6345.0       31725.0  Alhaji Musa  

Data Types:
transaction_id          str
transaction_date        str
market                  str
state                   str
product                 str
category                s

In [4]:
# Explore Weekly Prices Dataset
print("=== WEEKLY PRICES DATASET OVERVIEW ===\n")

print("First 5 rows:")
print(weekly_prices.head())
print("\nData Types:")
print(weekly_prices.dtypes)
print("\nMissing Values:")
print(weekly_prices.isnull().sum())
print("\nDuplicated Rows:", weekly_prices.duplicated().sum())

=== WEEKLY PRICES DATASET OVERVIEW ===

First 5 rows:
        product category  market week_start_date  avg_price
0  Palm Oil 25l      Oil  Ogbete      2024-01-15    34397.0
1     Rice 50KG   Grains    Wuse      2024-01-29    78517.0
2  Palm Oil 25L      Oil    Wuse      2024-01-08    33591.0
3   Beans Brown   Grains  Bodija      2024-02-05     3157.0
4     Rice 50kg   Grains    Wuse      2024-02-19    75891.0

Data Types:
product                str
category               str
market                 str
week_start_date        str
avg_price          float64
dtype: object

Missing Values:
product             0
category            0
market              0
week_start_date     0
avg_price          99
dtype: int64

Duplicated Rows: 21


---
## 4. Cleaning the Transactions Dataset

In [5]:
# Create a copy for cleaning
trans_clean = transactions.copy()

print("=== CLEANING TRANSACTIONS DATASET ===\n")

# 4.1 Standardize column names (lowercase, no spaces)
trans_clean.columns = trans_clean.columns.str.lower().str.replace(' ', '_')
print("1. Standardized column names")
print(f"   New columns: {list(trans_clean.columns)}\n")

# 4.2 Clean and standardize date formats
print("2. Cleaning date formats...")

# Function to parse various date formats
def parse_date(date_str):
    if pd.isna(date_str):
        return pd.NaT
    
    date_str = str(date_str).strip()
    
    # Try different formats
    formats = [
        '%Y/%m/%d',
        '%Y-%m-%d',
        '%d/%m/%Y',
    ]
    
    for fmt in formats:
        try:
            return pd.to_datetime(date_str, format=fmt)
        except:
            continue
    
    return pd.NaT

trans_clean['transaction_date'] = trans_clean['transaction_date'].apply(parse_date)
trans_clean['transaction_date'] = pd.to_datetime(trans_clean['transaction_date'], errors='coerce')

print(f"   Dates parsed. Missing dates: {trans_clean['transaction_date'].isnull().sum()}\n")

# 4.3 Clean product names (standardize casing)
print("3. Standardizing product names...")
trans_clean['product'] = trans_clean['product'].str.title()
print(f"   Unique products: {trans_clean['product'].unique()}\n")

# 4.4 Clean vendor names (remove leading/trailing spaces)
print("4. Cleaning vendor names...")
trans_clean['vendor'] = trans_clean['vendor'].str.strip()
print(f"   Unique vendors: {trans_clean['vendor'].unique()}\n")

# 4.5 Handle missing values in quantity and total_amount
print("5. Handling missing values...")

# Calculate total_amount where it is missing but quantity and unit_price exist
mask = trans_clean['total_amount'].isnull() & trans_clean['quantity'].notnull() & trans_clean['unit_price'].notnull()
trans_clean.loc[mask, 'total_amount'] = trans_clean.loc[mask, 'quantity'] * trans_clean.loc[mask, 'unit_price']

# Calculate quantity where it is missing but total_amount and unit_price exist
mask = trans_clean['quantity'].isnull() & trans_clean['total_amount'].notnull() & trans_clean['unit_price'].notnull()
trans_clean.loc[mask, 'quantity'] = trans_clean.loc[mask, 'total_amount'] / trans_clean.loc[mask, 'unit_price']

print(f"   Remaining missing quantity: {trans_clean['quantity'].isnull().sum()}")
print(f"   Remaining missing unit_price: {trans_clean['unit_price'].isnull().sum()}")
print(f"   Remaining missing total_amount: {trans_clean['total_amount'].isnull().sum()}\n")

# 4.6 Remove rows with critical missing values (no price or product info)
print("6. Removing rows with critical missing values...")
initial_rows = len(trans_clean)
trans_clean = trans_clean.dropna(subset=['product', 'market', 'state'])
print(f"   Rows removed: {initial_rows - len(trans_clean)}")
print(f"   Final shape: {trans_clean.shape}\n")

print("Transactions dataset cleaning complete!")

=== CLEANING TRANSACTIONS DATASET ===

1. Standardized column names
   New columns: ['transaction_id', 'transaction_date', 'market', 'state', 'product', 'category', 'quantity', 'unit_price', 'total_amount', 'vendor']

2. Cleaning date formats...
   Dates parsed. Missing dates: 97

3. Standardizing product names...
   Unique products: <StringArray>
['Rice 50Kg', 'Beans Brown', 'Garri White', 'Palm Oil 25L']
Length: 4, dtype: str

4. Cleaning vendor names...
   Unique vendors: <StringArray>
['Alhaji Musa', 'Mama Ngozi', 'Mr Ade', 'Mrs Okeke']
Length: 4, dtype: str

5. Handling missing values...
   Remaining missing quantity: 211
   Remaining missing unit_price: 62
   Remaining missing total_amount: 262

6. Removing rows with critical missing values...
   Rows removed: 0
   Final shape: (1000, 10)

Transactions dataset cleaning complete!


---
## 5. Cleaning the Weekly Prices Dataset

In [6]:
# Create a copy for cleaning
prices_clean = weekly_prices.copy()

print("=== CLEANING WEEKLY PRICES DATASET ===\n")

# 5.1 Standardize column names
prices_clean.columns = prices_clean.columns.str.lower().str.replace(' ', '_')
print("1. Standardized column names")
print(f"   New columns: {list(prices_clean.columns)}\n")

# 5.2 Convert week_start_date to datetime
print("2. Converting dates...")
prices_clean['week_start_date'] = pd.to_datetime(prices_clean['week_start_date'])
print(f"   Date range: {prices_clean['week_start_date'].min()} to {prices_clean['week_start_date'].max()}\n")

# 5.3 Standardize product names (match transaction dataset)
print("3. Standardizing product names...")
prices_clean['product'] = prices_clean['product'].str.title()

# Map product names to match transaction dataset
product_mapping = {
    'Palm Oil 25L': 'Palm Oil 25l',
    'Rice 50KG': 'Rice 50kg',
    'Beans Brown': 'Beans Brown',
    'Garri White': 'Garri White'
}
prices_clean['product'] = prices_clean['product'].replace(product_mapping)
print(f"   Unique products: {prices_clean['product'].unique()}\n")

# 5.4 Handle missing values in avg_price
print("4. Handling missing average prices...")
missing_before = prices_clean['avg_price'].isnull().sum()

# Fill missing avg_price with median for that product and market
prices_clean['avg_price'] = prices_clean.groupby(['product', 'market'])['avg_price'].transform(
    lambda x: x.fillna(x.median())
)

# If still missing, fill with product median
prices_clean['avg_price'] = prices_clean.groupby('product')['avg_price'].transform(
    lambda x: x.fillna(x.median())
)

missing_after = prices_clean['avg_price'].isnull().sum()
print(f"   Missing values filled: {missing_before - missing_after}")
print(f"   Remaining missing: {missing_after}\n")

# 5.5 Remove rows where avg_price is still missing
initial_rows = len(prices_clean)
prices_clean = prices_clean.dropna(subset=['avg_price'])
print(f"5. Rows removed due to missing avg_price: {initial_rows - len(prices_clean)}")
print(f"   Final shape: {prices_clean.shape}\n")

print("Weekly prices dataset cleaning complete!")

=== CLEANING WEEKLY PRICES DATASET ===

1. Standardized column names
   New columns: ['product', 'category', 'market', 'week_start_date', 'avg_price']

2. Converting dates...
   Date range: 2024-01-01 00:00:00 to 2024-03-04 00:00:00

3. Standardizing product names...
   Unique products: <StringArray>
['Palm Oil 25l', 'Rice 50Kg', 'Beans Brown', 'Garri White']
Length: 4, dtype: str

4. Handling missing average prices...
   Missing values filled: 99
   Remaining missing: 0

5. Rows removed due to missing avg_price: 0
   Final shape: (1000, 5)

Weekly prices dataset cleaning complete!


---
## 6. Merging the Datasets

In [7]:
print("=== MERGING DATASETS ===\n")

# First, let us see the common columns
print("Transactions columns:", trans_clean.columns.tolist())
print("Prices columns:", prices_clean.columns.tolist())
print()

# Check unique values for merging keys
print("Unique products in transactions:", trans_clean['product'].nunique())
print("Unique products in prices:", prices_clean['product'].nunique())
print()
print("Unique markets in transactions:", trans_clean['market'].nunique())
print("Unique markets in prices:", prices_clean['market'].nunique())
print()

# Common products and markets
common_products = set(trans_clean['product'].unique()) & set(prices_clean['product'].unique())
common_markets = set(trans_clean['market'].unique()) & set(prices_clean['market'].unique())
print(f"Common products: {common_products}")
print(f"Common markets: {common_markets}")

=== MERGING DATASETS ===

Transactions columns: ['transaction_id', 'transaction_date', 'market', 'state', 'product', 'category', 'quantity', 'unit_price', 'total_amount', 'vendor']
Prices columns: ['product', 'category', 'market', 'week_start_date', 'avg_price']

Unique products in transactions: 4
Unique products in prices: 4

Unique markets in transactions: 4
Unique markets in prices: 4

Common products: {'Beans Brown', 'Garri White', 'Rice 50Kg'}
Common markets: {'Wuse', 'Balogun', 'Ogbete', 'Bodija'}


In [8]:
# For merging, we will use product, market, and date proximity
# First, add a 'week_start_date' column to transactions based on transaction_date
trans_clean['week_start_date'] = trans_clean['transaction_date'] - pd.to_timedelta(
    trans_clean['transaction_date'].dt.dayofweek, unit='d'
)

# Perform the merge on product, market, and week_start_date
merged_data = pd.merge(
    trans_clean,
    prices_clean,
    on=['product', 'market', 'week_start_date'],
    how='left',
    suffixes=('', '_weekly')
)

print(f"Merge complete!")
print(f"Merged dataset shape: {merged_data.shape}")
print(f"\nColumns in merged dataset:")
print(merged_data.columns.tolist())

Merge complete!
Merged dataset shape: (4432, 13)

Columns in merged dataset:
['transaction_id', 'transaction_date', 'market', 'state', 'product', 'category', 'quantity', 'unit_price', 'total_amount', 'vendor', 'week_start_date', 'category_weekly', 'avg_price']


In [9]:
# Check merge success
print("\nMerge Statistics:")
print(f"Total transactions: {len(merged_data)}")
print(f"Transactions with weekly price match: {merged_data['avg_price'].notnull().sum()}")
print(f"Transactions without weekly price match: {merged_data['avg_price'].isnull().sum()}")
print(f"Match rate: {merged_data['avg_price'].notnull().mean()*100:.1f}%")


Merge Statistics:
Total transactions: 4432
Transactions with weekly price match: 4087
Transactions without weekly price match: 345
Match rate: 92.2%


---
## 7. Handling Missing Values in Merged Data

In [10]:
print("=== HANDLING MISSING VALUES IN MERGED DATA ===\n")

# 7.1 For transactions without weekly price match, use the transaction unit_price as reference
print("1. Filling missing weekly avg_price with transaction unit_price...")
merged_data['avg_price'] = merged_data['avg_price'].fillna(merged_data['unit_price'])

# 7.2 Handle any remaining missing categorical values
print("2. Filling missing categorical values...")
merged_data['category'] = merged_data['category'].fillna('Unknown')

# 7.3 Drop rows with critical missing values
print("3. Dropping rows with critical missing values...")
initial_count = len(merged_data)
merged_data = merged_data.dropna(subset=['transaction_date', 'product', 'market'])
print(f"   Dropped {initial_count - len(merged_data)} rows")

# 7.4 Final missing values check
print("\n4. Final missing values summary:")
print(merged_data.isnull().sum())

=== HANDLING MISSING VALUES IN MERGED DATA ===

1. Filling missing weekly avg_price with transaction unit_price...
2. Filling missing categorical values...
3. Dropping rows with critical missing values...
   Dropped 97 rows

4. Final missing values summary:
transaction_id         0
transaction_date       0
market                 0
state                  0
product                0
category               0
quantity             960
unit_price           256
total_amount        1164
vendor                 0
week_start_date        0
category_weekly      248
avg_price             17
dtype: int64


---
## 8. Reshaping the Dataset

In [11]:
print("=== RESHAPING THE DATASET ===\n")

# 8.1 Create new features for analysis
print("1. Creating new features...")

# Extract date components
merged_data['year'] = merged_data['transaction_date'].dt.year
merged_data['month'] = merged_data['transaction_date'].dt.month
merged_data['day_of_week'] = merged_data['transaction_date'].dt.day_name()
merged_data['week_of_year'] = merged_data['transaction_date'].dt.isocalendar().week

# Calculate price difference between transaction price and weekly average
merged_data['price_diff'] = merged_data['unit_price'] - merged_data['avg_price']
merged_data['price_diff_pct'] = (merged_data['price_diff'] / merged_data['avg_price']) * 100

# Create price category
merged_data['price_category'] = pd.cut(
    merged_data['unit_price'],
    bins=[0, 10000, 30000, 50000, float('inf')],
    labels=['Low', 'Medium', 'High', 'Premium']
)

print("   Created: year, month, day_of_week, week_of_year")
print("   Created: price_diff, price_diff_pct")
print("   Created: price_category\n")

# 8.2 Reorder columns for better readability
print("2. Reordering columns...")
column_order = [
    'transaction_id', 'transaction_date', 'year', 'month', 'week_of_year', 'day_of_week',
    'market', 'state', 'product', 'category', 'vendor',
    'quantity', 'unit_price', 'avg_price', 'total_amount',
    'price_diff', 'price_diff_pct', 'price_category',
    'week_start_date'
]

merged_data = merged_data[column_order]
print(f"   Dataset shape: {merged_data.shape}\n")

# 8.3 Display sample of reshaped data
print("3. Sample of reshaped data:")
print(merged_data.head())

=== RESHAPING THE DATASET ===

1. Creating new features...
   Created: year, month, day_of_week, week_of_year
   Created: price_diff, price_diff_pct
   Created: price_category

2. Reordering columns...
   Dataset shape: (4335, 19)

3. Sample of reshaped data:
  transaction_id transaction_date  year  month  week_of_year day_of_week  \
0        TXN0001       2024-01-24  2024      1             4   Wednesday   
1        TXN0001       2024-01-24  2024      1             4   Wednesday   
2        TXN0001       2024-01-24  2024      1             4   Wednesday   
3        TXN0002       2024-01-17  2024      1             3   Wednesday   
4        TXN0002       2024-01-17  2024      1             3   Wednesday   

    market  state    product category       vendor  quantity  unit_price  \
0  Balogun  Lagos  Rice 50Kg   Grains  Alhaji Musa       5.0     78006.0   
1  Balogun  Lagos  Rice 50Kg   Grains  Alhaji Musa       5.0     78006.0   
2  Balogun  Lagos  Rice 50Kg   Grains  Alhaji Musa     

---
## 9. Final Data Validation

In [12]:
print("=== FINAL DATA VALIDATION ===\n")

# 9.1 Check for duplicates
duplicates = merged_data.duplicated().sum()
print(f"1. Duplicate rows: {duplicates}")
if duplicates > 0:
    merged_data = merged_data.drop_duplicates()
    print(f"   Removed {duplicates} duplicates")

# 9.2 Data type validation
print("\n2. Data types:")
print(merged_data.dtypes)

# 9.3 Value ranges validation
print("\n3. Value ranges:")
print(f"   Quantity: {merged_data['quantity'].min():.2f} to {merged_data['quantity'].max():.2f}")
print(f"   Unit Price: N{merged_data['unit_price'].min():.2f} to N{merged_data['unit_price'].max():.2f}")
print(f"   Total Amount: N{merged_data['total_amount'].min():.2f} to N{merged_data['total_amount'].max():.2f}")

# 9.4 Check for logical inconsistencies
print("\n4. Logical consistency checks:")
# Check if quantity * unit_price approximately equals total_amount
calculated_total = merged_data['quantity'] * merged_data['unit_price']
mismatch = np.abs(calculated_total - merged_data['total_amount']) > 1
print(f"   Records with quantity * price not equal to total: {mismatch.sum()}")

# 9.5 Final statistics
print("\n5. Final dataset statistics:")
print(f"   Total records: {len(merged_data)}")
print(f"   Date range: {merged_data['transaction_date'].min()} to {merged_data['transaction_date'].max()}")
print(f"   Products: {merged_data['product'].nunique()}")
print(f"   Markets: {merged_data['market'].nunique()}")
print(f"   States: {merged_data['state'].nunique()}")
print(f"   Vendors: {merged_data['vendor'].nunique()}")

=== FINAL DATA VALIDATION ===

1. Duplicate rows: 171
   Removed 171 duplicates

2. Data types:
transaction_id                 str
transaction_date    datetime64[us]
year                         int32
month                        int32
week_of_year                UInt32
day_of_week                    str
market                         str
state                          str
product                        str
category                       str
vendor                         str
quantity                   float64
unit_price                 float64
avg_price                  float64
total_amount               float64
price_diff                 float64
price_diff_pct             float64
price_category            category
week_start_date     datetime64[us]
dtype: object

3. Value ranges:
   Quantity: 1.00 to 10.00
   Unit Price: N1411.00 to N82957.00
   Total Amount: N1496.00 to N828880.00

4. Logical consistency checks:
   Records with quantity * price not equal to total: 0

5. Final datase

In [13]:
# Quick summary statistics
print("\n=== SUMMARY STATISTICS ===\n")
print(merged_data.describe())


=== SUMMARY STATISTICS ===

                 transaction_date    year   month  week_of_year     quantity  \
count                        4164  4164.0  4164.0        4164.0  3245.000000   
mean   2024-01-16 10:19:42.708933  2024.0     1.0      2.798991     4.785208   
min           2024-01-01 00:00:00  2024.0     1.0           1.0     1.000000   
25%           2024-01-09 00:00:00  2024.0     1.0           2.0     2.000000   
50%           2024-01-17 00:00:00  2024.0     1.0           3.0     5.000000   
75%           2024-01-24 00:00:00  2024.0     1.0           4.0    10.000000   
max           2024-01-31 00:00:00  2024.0     1.0           5.0    10.000000   
std                           NaN     0.0     0.0      1.259769     3.574006   

         unit_price     avg_price   total_amount   price_diff  price_diff_pct  \
count   3919.000000   4147.000000    3049.000000  3919.000000     3919.000000   
mean   33009.102832  32537.413311  152111.419810   394.906736       20.010112   
min    

---
## 10. Exporting the Cleaned Dataset

In [14]:
print("=== EXPORTING CLEANED DATASET ===\n")

# Export to CSV
output_filename = 'nigerian_market_cleaned.csv'
merged_data.to_csv(output_filename, index=False)

print(f"Cleaned dataset exported to: {output_filename}")
print(f"   File size: {merged_data.memory_usage(deep=True).sum() / 1024:.2f} KB")
print(f"   Total rows: {len(merged_data)}")
print(f"   Total columns: {len(merged_data.columns)}")
print(f"\nColumns exported:")
for i, col in enumerate(merged_data.columns, 1):
    print(f"   {i}. {col}")

=== EXPORTING CLEANED DATASET ===

Cleaned dataset exported to: nigerian_market_cleaned.csv
   File size: 1945.98 KB
   Total rows: 4164
   Total columns: 19

Columns exported:
   1. transaction_id
   2. transaction_date
   3. year
   4. month
   5. week_of_year
   6. day_of_week
   7. market
   8. state
   9. product
   10. category
   11. vendor
   12. quantity
   13. unit_price
   14. avg_price
   15. total_amount
   16. price_diff
   17. price_diff_pct
   18. price_category
   19. week_start_date
